## Project OFH participants into HGDP + 1KG space

This .ipynb script takes the allele frequencies (https://console.cloud.google.com/storage/browser/_details/covid19-hg-public/pca_projection/hgdp_tgp_pca_covid19hgi_snps_loadings.GRCh38.plink.afreq), loadings (https://console.cloud.google.com/storage/browser/_details/covid19-hg-public/pca_projection/hgdp_tgp_pca_covid19hgi_snps_loadings.GRCh38.plink.tsv), both for GRCh38 and rsids (https://console.cloud.google.com/storage/browser/_details/covid19-hg-public/pca_projection/hgdp_tgp_pca_covid19hgi_snps_loadings.rsid.plink.tsv;tab=live_object and https://console.cloud.google.com/storage/browser/_details/covid19-hg-public/pca_projection/hgdp_tgp_pca_covid19hgi_snps_loadings.rsid.plink.afreq;tab=live_object) from the Human Genome Diversity Project + 1,000 Genomes project, and projects Our Future Health participants into that principal component space. 

Running this script requires the following programs:

- plink2
- bgenix

You can set filepaths in the preferences chunk below, then run the rest of the script.

In [ ]:
## User preferences

hgdp_1kg_GRCh38_afreq_filepath <- "project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/HGDP_1KG/pca_projection_hgdp_tgp_pca_covid19hgi_snps_loadings.GRCh38.plink.afreq"
hgdp_1kg_GRCh38_loadings_filepath <- "project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/HGDP_1KG/pca_projection_hgdp_tgp_pca_covid19hgi_snps_loadings.GRCh38.plink.tsv"
hgdp_1kg_rsid_afreq_filepath <- "project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/HGDP_1KG/pca_projection_hgdp_tgp_pca_covid19hgi_snps_loadings.rsid.plink.afreq"
hgdp_1kg_rsid_loadings_filepath <- "project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/HGDP_1KG/pca_projection_hgdp_tgp_pca_covid19hgi_snps_loadings.rsid.plink.tsv"
programs_directory <- "project-J8PJg1V25y16J0Y3F05pPJBj:/programs/"
bgen_directory <- "project-J8PJg1V25y16J0Y3F05pPJBj:/imputed_bgen/"

In [ ]:
#############################################################
#==== Step 0 - load packages and set up helper functions ====#
##############################################################

required_packages <- c("data.table")

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }#
}

run_command <- function(command, args) {
  shell_command <- paste(
    c(shQuote(command, type = "sh"), vapply(args, shQuote, character(1), type = "sh")),
    collapse = " "
  )
  bash_command <- paste(
    "unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy;",
    shell_command
  )

  result <- system2(
    "bash",
    args = c("-c", shQuote(bash_command, type = "sh")),
    stdout = TRUE,
    stderr = TRUE
  )

  status <- attr(result, "status")
  if (is.null(status)) status <- 0L

  if (status != 0L) {
    stop(
      command, " command failed:\n",
      paste(result, collapse = "\n"),
      call. = FALSE
    )
  }

  result
}

run_dx <- function(args) run_command("dx", args)

In [ ]:
################################################
#== Step 1 - Fetch HGDP+1KG projection files ==#
################################################

# Download PLINK2 if necessary
if (!file.exists("plink2")) {
  run_dx(c(
    "download",
    paste0(programs_directory, "plink2")
  ))
}

if (!file.exists("plink2")) {
  stop("plink2 was not downloaded from the programs directory.")
}

# Make the downloaded PLINK2 binary executable
plink2_bin <- normalizePath(
  "plink2",
  mustWork = TRUE
)

Sys.chmod(
  plink2_bin,
  mode = "0755"
)

# Confirm executable permissions
plink2_info <- file.info(plink2_bin)

message("Using PLINK2 at: ", plink2_bin)
message(
  "PLINK2 permissions: ",
  as.character(plink2_info$mode)
)

# Test the executable before processing any BGEN shards
plink2_test <- system2(
  plink2_bin,
  "--version",
  stdout = TRUE,
  stderr = TRUE
)

plink2_status <- attr(plink2_test, "status")

if (!is.null(plink2_status) && plink2_status != 0L) {
  cat(
    "---- PLINK2 executable test ----\n",
    paste(plink2_test, collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(
    sprintf(
      "PLINK2 exists but could not be executed; exit status %d.",
      plink2_status
    )
  )
}

message(
  "PLINK2 executable test passed: ",
  paste(plink2_test, collapse = " ")
)

# Fetch HGDP+1KG projection assets into the current JupyterLab session

if (!file.exists(basename(hgdp_1kg_GRCh38_afreq_filepath))) {
  run_dx(c("download", paste0(hgdp_1kg_GRCh38_afreq_filepath)))
}

if (!file.exists(basename(hgdp_1kg_GRCh38_loadings_filepath))) {
  run_dx(c("download", paste0(hgdp_1kg_GRCh38_loadings_filepath)))
}
if (!file.exists(basename(hgdp_1kg_rsid_afreq_filepath))) {
  run_dx(c("download", paste0(hgdp_1kg_rsid_afreq_filepath)))
}

if (!file.exists(basename(hgdp_1kg_rsid_loadings_filepath))) {
  run_dx(c("download", paste0(hgdp_1kg_rsid_loadings_filepath)))
}


GRCh38_afreq_file   <- basename(hgdp_1kg_GRCh38_afreq_filepath)
GRCh38_loadings_tsv <- basename(hgdp_1kg_GRCh38_loadings_filepath)
rsid_afreq <- basename(hgdp_1kg_rsid_afreq_filepath)
rsid_loadings <- basename(hgdp_1kg_rsid_loadings_filepath)

stopifnot(
  file.exists(GRCh38_afreq_file),
  file.exists(GRCh38_loadings_tsv),
  file.exists(rsid_loadings),
  file.exists(rsid_afreq)
)

# Make a one-column GRCh38 position list

GRCh38_loadings <- data.table::fread(
  GRCh38_loadings_tsv,
  data.table = TRUE
)

required_columns <- c("ID", "ALT")

missing_columns <- setdiff(
  required_columns,
  names(GRCh38_loadings)
)

if (length(missing_columns)) {
  stop(
    sprintf(
      "GRCh38 loadings file is missing required column(s): %s",
      paste(missing_columns, collapse = ", ")
    )
  )
}

# Parse IDs such as chr1:123456:A:G
id_parts <- data.table::tstrsplit(
  GRCh38_loadings$ID,
  ":",
  fixed = TRUE
)

if (length(id_parts) != 4L) {
  stop(
    paste0(
      "The GRCh38 loading IDs do not consistently have four ",
      "colon-separated fields: chr, position, REF, ALT."
    )
  )
}

GRCh38_loadings[, CHROM := sub(
  "^chr",
  "",
  id_parts[[1]],
  ignore.case = TRUE
)]

GRCh38_loadings[, POS := id_parts[[2]]]
GRCh38_loadings[, REF_FROM_ID := toupper(id_parts[[3]])]
GRCh38_loadings[, ALT_FROM_ID := toupper(id_parts[[4]])]
GRCh38_loadings[, ALT := toupper(ALT)]

## validate the ID against the ALT column

alt_mismatch <- GRCh38_loadings[
  !is.na(ALT) &
  !is.na(ALT_FROM_ID) &
  ALT != ALT_FROM_ID
]

if (nrow(alt_mismatch)) {
  cat(
    "Examples of ID/ALT mismatches:\n"
  )

  print(
    head(
      alt_mismatch[, .(
        ID,
        ALT,
        REF_FROM_ID,
        ALT_FROM_ID
      )],
      10L
    )
  )

  stop(sprintf(
    paste0(
      "%d loading rows have an ALT column that does not match ",
      "the fourth component of ID."
    ),
    nrow(alt_mismatch)
  ))
}

message(
  "GRCh38 loading IDs are consistent with chr:position:REF:ALT."
)

# Construct both possible orientations of the GRCh38 loadings for PLINK2 scoring

# OFH-style CHROM-POS-REF-ALT
GRCh38_loadings[, OFH_ID_REF_ALT := paste(
  CHROM,
  POS,
  REF_FROM_ID,
  ALT_FROM_ID,
  sep = "-"
)]

# Also include CHROM-POS-ALT-REF because some OFH entries may use
# the opposite allele order
GRCh38_loadings[, OFH_ID_ALT_REF := paste(
  CHROM,
  POS,
  ALT_FROM_ID,
  REF_FROM_ID,
  sep = "-"
)]

ofh_extraction_ids <- unique(
  c(
    GRCh38_loadings$OFH_ID_REF_ALT,
    GRCh38_loadings$OFH_ID_ALT_REF
  )
)

ofh_extraction_ids <- ofh_extraction_ids[
  !is.na(ofh_extraction_ids) &
  nzchar(ofh_extraction_ids)
]

if (!length(ofh_extraction_ids)) {
  stop(
    "No OFH-formatted positional identifiers were generated."
  )
}

writeLines(
  ofh_extraction_ids,
  "variants.ofh_id"
)

message(sprintf(
  paste0(
    "Wrote %d unique OFH-formatted identifiers ",
    "from %d GRCh38 loading variants."
  ),
  length(ofh_extraction_ids),
  data.table::uniqueN(GRCh38_loadings$ID)
))

message("First OFH extraction identifiers:")

print(
  head(
    ofh_extraction_ids,
    10L
  )
)

In [ ]:
#######################################################################
# Step 2A — Subset OFH PGENs to SNPs used to calculate HGDP+1KG PCs   #
# (bgenix subset → index → pgen)                                      #
#######################################################################

stopifnot(
  file.exists(GRCh38_loadings_tsv),
  file.exists(GRCh38_afreq_file),
  file.exists("variants.ofh_id"),
  file.size("variants.ofh_id") > 0
)

message("First GRCh38 IDs in extraction file:")

print(
  head(
    readLines("variants.ofh_id"),
    10
  )
)

### Load in bgenix 

if (!file.exists("bgenix")) {
  run_dx(c("download", paste0(programs_directory, "bgenix")))
}

if (!file.exists("bgenix")) stop("bgenix was not downloaded from the programs directory.")

# The downloaded file is the executable itself, not a source archive.
bgenix_bin <- normalizePath("bgenix", mustWork = TRUE)
Sys.chmod(bgenix_bin, "0755")
Sys.setenv(PATH = paste(dirname(bgenix_bin), Sys.getenv("PATH"), sep = ":"))

bgenix_on_path <- Sys.which("bgenix")
if (!nzchar(bgenix_on_path)) stop("bgenix is not executable or is not available on PATH.")
message("Using bgenix at ", bgenix_on_path)
system2(bgenix_on_path, "-help")


# Helper: choose a writable work directory with an actual write test
choose_writable_dir <- function(candidates) {
  # return first path for which we can create+delete a file
  for (d in candidates) {
    if (!nzchar(d)) next
    ok <- try(dir.create(d, recursive = TRUE, showWarnings = FALSE), silent = TRUE)
    if (!dir.exists(d)) next
    tf <- file.path(d, sprintf(".writetest_%s", as.integer(runif(1, 1e6, 9e6))))
    can <- tryCatch({ file.create(tf) }, warning = function(e) FALSE, error = function(e) FALSE)
    if (isTRUE(can)) { file.remove(tf); return(normalizePath(d, mustWork = TRUE)) }
  }
  return(NA_character_)
}

# Prefer project subfolders if writable; otherwise fall back to CWD/home/tempdir.
job_tag <- Sys.getenv("DX_JOB_ID", unset = format(Sys.time(), "%Y%m%d_%H%M%S"))
work_dir <- choose_writable_dir(c(
  file.path("/mnt/project", "work", paste0("pca_subset_", job_tag)),
  file.path("/mnt/project", paste0("pca_subset_", job_tag)),
  file.path(getwd(),       paste0("pca_subset_", job_tag)),
  file.path("~",           paste0("pca_subset_", job_tag)),
  tempdir()  # last resort
))
if (is.na(work_dir)) {
  stop("Could not find a writable work directory. Check project permissions.")
}
message("Using work_dir: ", work_dir)

# For downstream parts, reuse 'work_dir' consistently instead of Sys.getenv("TMPDIR")

tmp_dir <- work_dir

message("Using tmp_dir: ", tmp_dir)

# Ensure tmp_dir exists and is writable; if not, fall back to a fresh temp path

dir.create(tmp_dir, recursive = TRUE, showWarnings = FALSE)

.testfile <- file.path(tmp_dir, sprintf(".writetest_%s", as.integer(Sys.time())))
.canwrite <- tryCatch(isTRUE(file.create(.testfile)),
                      warning = function(e) FALSE, error = function(e) FALSE)
if (.canwrite) unlink(.testfile, force = TRUE) else {
  job_tag <- Sys.getenv("DX_JOB_ID", unset = format(Sys.time(), "%Y%m%d_%H%M%S"))
  tmp_dir <- file.path(tempdir(), paste0("pca_subset_", job_tag))
  dir.create(tmp_dir, recursive = TRUE, showWarnings = FALSE)
  
  .testfile <- file.path(tmp_dir, sprintf(".writetest_%s", as.integer(Sys.time())))
  .canwrite <- tryCatch(isTRUE(file.create(.testfile)),
                        warning = function(e) FALSE, error = function(e) FALSE)
  if (.canwrite) unlink(.testfile, force = TRUE) else {
    stop("No writable tmp_dir available. Check filesystem permissions.")
  }
}
message("Final tmp_dir (writable): ", tmp_dir)

# Define per‑loop helpers here so they’re in scope

extract_file <- normalizePath(
  "variants.ofh_id"
)

subset_prefixes <- character()

threads <- "16"
mem_gb <- "64"

all_files <- run_dx(c("ls", bgen_directory))

read_bgen_variant_count <- function(bgen_file) {
  con <- file(
    bgen_file,
    open = "rb"
  )

  on.exit(close(con))

  header_values <- readBin(
    con,
    what = integer(),
    n = 4L,
    size = 4L,
    endian = "little"
  )

  if (length(header_values) != 4L) {
    stop(
      "Could not read the BGEN header from: ",
      bgen_file
    )
  }

  # BGEN header values:
  # 1. offset
  # 2. header-block length
  # 3. number of variants
  # 4. number of samples
  as.numeric(header_values[3L])
}

for (chr in 1:22) {

  chr_bgens <- sort(grep(
    paste0(
      "^ofh_imputed\\.v6\\.chr",
      chr,
      "-b[0-9]+\\.bgen$"
    ),
    all_files,
    value = TRUE
  ))

  message(sprintf(
    "Chromosome %d: %d shards found",
    chr,
    length(chr_bgens)
  ))

  if (!length(chr_bgens)) {
    warning(sprintf(
      "No BGEN shards found for chromosome %d",
      chr
    ))
    next
  }

  message(sprintf(
    "Full run: processing all %d chromosome %d shards",
    length(chr_bgens),
    chr
  ))

for (src_bgen_name in chr_bgens) {

  src_sample_name <- sub(
    "\\.bgen$",
    ".sample",
    src_bgen_name
  )

  src_bgi_name <- paste0(
    src_bgen_name,
    ".bgi"
  )

  shard_name <- sub(
    "\\.bgen$",
    "",
    basename(src_bgen_name)
  )

  # Put downloaded source files in tmp_dir
  local_bgen <- file.path(
    tmp_dir,
    basename(src_bgen_name)
  )

  local_bgi <- paste0(
    local_bgen,
    ".bgi"
  )

  local_sample <- file.path(
    tmp_dir,
    basename(src_sample_name)
  )

  message(
    sprintf(
      "[chr %d] Downloading source files for %s",
      chr,
      shard_name
    )
  )

  # Remove files left by an earlier interrupted attempt
  unlink(
    c(
      local_bgen,
      local_bgi,
      local_sample
    ),
    force = TRUE
  )

  run_dx(c(
    "download",
    paste0(bgen_directory, src_bgen_name),
    "-o",
    local_bgen
  ))

  run_dx(c(
    "download",
    paste0(bgen_directory, src_bgi_name),
    "-o",
    local_bgi
  ))

  run_dx(c(
    "download",
    paste0(bgen_directory, src_sample_name),
    "-o",
    local_sample
  ))

  required_source_files <- c(
    local_bgen,
    local_bgi,
    local_sample
  )

  missing_source_files <- required_source_files[
    !file.exists(required_source_files)
  ]

  if (length(missing_source_files)) {
    stop(
      sprintf(
        "Failed to download required source file(s) for %s: %s",
        shard_name,
        paste(missing_source_files, collapse = ", ")
      )
    )
  }

## check first few records from the downloaded chromosome shard

if (chr == 1L && identical(src_bgen_name, chr_bgens[1L])) {

  preview_cmd <- sprintf(
    "%s -g %s -list 2>&1 | head -n 20",
    shQuote(bgenix_on_path),
    shQuote(local_bgen)
  )

  raw_index_preview <- system(
    preview_cmd,
    intern = TRUE
  )

  cat(
    "---- First unfiltered OFH BGEN index lines ----\n",
    paste(raw_index_preview, collapse = "\n"),
    "\n",
    sep = ""
  )

  cat(
    "---- First requested GRCh38 positional IDs ----\n",
    paste(
      head(readLines(extract_file), 10L),
      collapse = "\n"
    ),
    "\n",
    sep = ""
  )
}

## Precheck: determine exactly how many requested variants are in this shard

match_out <- suppressWarnings(system2(
  bgenix_on_path,
  c(
    "-g", local_bgen,
    "-list",
    "-incl-rsids", extract_file
  ),
  stdout = TRUE,
  stderr = TRUE
))

match_status <- attr(match_out, "status")

if (!is.null(match_status) && match_status != 0L) {
  cat(
    "---- bgenix precheck output ----\n",
    paste(match_out, collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(sprintf(
    "bgenix precheck failed for chr %d, shard %s.",
    chr,
    shard_name
  ))
}

# bgenix reports a line resembling:
# "# bgenix: success, total 0 variants."

summary_line <- grep(
  "total[[:space:]]+[0-9]+[[:space:]]+variants",
  match_out,
  value = TRUE
)

if (!length(summary_line)) {
  cat(
    "---- Unexpected bgenix precheck output ----\n",
    paste(match_out, collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(sprintf(
    paste0(
      "Could not determine the number of matching variants ",
      "for chr %d, shard %s."
    ),
    chr,
    shard_name
  ))
}

# Use the final matching summary line in case bgenix emitted more than one
summary_line <- tail(summary_line, 1L)

n_matching_variants <- as.integer(sub(
  ".*total[[:space:]]+([0-9]+)[[:space:]]+variants.*",
  "\\1",
  summary_line
))

if (is.na(n_matching_variants)) {
  cat(
    "---- Unparseable bgenix summary ----\n",
    summary_line,
    "\n",
    sep = ""
  )

  stop(sprintf(
    "Could not parse variant count for chr %d, shard %s.",
    chr,
    shard_name
  ))
}

message(sprintf(
  "[chr %d] %s: %d requested variants found",
  chr,
  shard_name,
  n_matching_variants
))

if (n_matching_variants == 0L) {
  message(sprintf(
    "[chr %d] No requested variants found in %s; skipping.",
    chr,
    shard_name
  ))

  unlink(
    c(
      local_bgen,
      local_bgi,
      local_sample
    ),
    force = TRUE
  )

  next
}

    subset_bgen <- file.path(
        tmp_dir,
        paste0(shard_name, "_subset.bgen")
    )

    unlink(
      c(
        subset_bgen,
        paste0(subset_bgen, ".bgi")
      ),
      force = TRUE
    )
    
    subset_err <- paste0(
      subset_bgen,
      ".stderr"
    )

    unlink(
      subset_err,
      force = TRUE
    )

    cmd_subset <- sprintf(
      "%s -g %s -incl-rsids %s > %s 2> %s",
      shQuote(bgenix_on_path),
      shQuote(local_bgen),
      shQuote(extract_file),
      shQuote(subset_bgen),
      shQuote(subset_err)
    )

    message(sprintf(
      "[chr %d] Extracting %d variants from %s",
      chr,
      n_matching_variants,
      shard_name
    ))

    st1 <- system(cmd_subset)

    if (
      st1 != 0L ||
      !file.exists(subset_bgen) ||
      file.size(subset_bgen) == 0
    ) {
      if (file.exists(subset_err)) {
        cat(
          "---- bgenix stderr ----\n",
          paste(
            readLines(subset_err, warn = FALSE),
            collapse = "\n"
          ),
          "\n",
          sep = ""
        )
      }

      stop(sprintf(
        "bgenix extraction failed for chr %d, shard %s.",
        chr,
        shard_name
      ))
    }

    # A BGEN containing zero variants still has a nonzero-sized header,
    # so inspect the variant count explicitly.
    subset_variant_count <- read_bgen_variant_count(
      subset_bgen
    )

    message(sprintf(
      "[chr %d] %s: extracted BGEN header reports %d variants",
      chr,
      shard_name,
      subset_variant_count
    ))

    if (subset_variant_count == 0L) {
      message(sprintf(
        "[chr %d] Extracted BGEN for %s is empty; skipping.",
        chr,
        shard_name
      ))

      unlink(
        c(
          local_bgen,
          local_bgi,
          local_sample,
          subset_bgen,
          subset_err
        ),
        force = TRUE
      )

      next
    }

    if (subset_variant_count != n_matching_variants) {
      warning(sprintf(
        paste0(
          "Variant-count mismatch for chr %d, shard %s: ",
          "precheck found %d but extracted BGEN reports %d."
        ),
        chr,
        shard_name,
        n_matching_variants,
        subset_variant_count
      ))
    }
    
    # Convert the small BGEN to PGEN (use the staged .sample)
    pfx <- file.path(
        tmp_dir,
        paste0(shard_name, "_subset")
    )

    unlink(
      paste0(
        pfx,
        c(
          ".pgen",
          ".pvar",
          ".psam",
          ".log"
        )
      ),
      force = TRUE
    )

    args_make_pgen <- c(
      "--threads", threads,
      "--memory",  as.character(as.integer(mem_gb) * 1024),
      "--bgen",    subset_bgen, "ref-first",
      "--sample",  local_sample,
      "--make-pgen",
      "--out",     pfx
    )
    
    message(
      sprintf(
        "[chr %d] %s -> PGEN",
        chr,
        basename(src_bgen_name)
      )
    )

out_txt <- system2(
  plink2_bin,
  args_make_pgen,
  stdout = TRUE,
  stderr = TRUE
)

required_pgen_files <- paste0(
  pfx,
  c(".pgen", ".pvar", ".psam")
)

if (!all(file.exists(required_pgen_files))) {
  logf <- paste0(pfx, ".log")

  if (file.exists(logf)) {
    cat(
      "---- PLINK log tail ----\n",
      paste(tail(readLines(logf), 50), collapse = "\n"),
      "\n",
      sep = ""
    )
  }

  cat(
    "---- Captured PLINK output ----\n",
    paste(tail(out_txt, 50), collapse = "\n"),
    "\n",
    sep = ""
  )

  stop(
    sprintf(
      "PLINK --make-pgen failed for chr %d, shard %s.",
      chr,
      shard_name
    )
  )
}

    # Only record a successfully completed PGEN fileset
    subset_prefixes <- c(
      subset_prefixes,
      pfx
    )

    # Reclaim disk space before downloading the next source shard
    unlink(
      c(
        local_bgen,
        local_bgi,
        local_sample,
        subset_bgen,
        paste0(subset_bgen, ".bgi"),
        subset_err
      ),
      force = TRUE
    )

  } # end shard loop
} # end chromosome loop

# Optional: you can delete subset_bgen + .bgi now to reclaim space
# unlink(c(subset_bgen, paste0(subset_bgen, ".bgi")), force = TRUE)

system("df -h")
system(sprintf("df -h %s", shQuote(work_dir)))

## count total pgens produced

message(sprintf(
  "Total shard-level PGENs produced: %d",
  length(subset_prefixes)
))

if (anyDuplicated(subset_prefixes)) {
  duplicated_prefixes <- unique(
    subset_prefixes[duplicated(subset_prefixes)]
  )

  stop(sprintf(
    "Duplicate shard-level PGEN prefixes detected: %s",
    paste(duplicated_prefixes, collapse = ", ")
  ))
}




In [ ]:
# ===================================== #
# Part 2B — Merge to a single `.pfile`  #
# ===================================== #

merge_list <- file.path(tmp_dir, "pmerge_list.txt")

# Which per-chr pfiles exist?

complete_fileset <- vapply(
  subset_prefixes,
  function(pfx) {
    all(file.exists(paste0(
      pfx,
      c(".pgen", ".pvar", ".psam")
    )))
  },
  logical(1)
)

existing <- which(complete_fileset)

if (!length(existing)) {
  stop(
    "No complete shard-level PGEN filesets were produced. Check Step 2A logs."
  )
}

if (any(!complete_fileset)) {
  warning(sprintf(
    "%d shard-level PGEN prefixes are incomplete and will be excluded.",
    sum(!complete_fileset)
  ))
}

if (length(existing) == 1L) {
  stop(
    paste0(
      "Only one shard-level PGEN was produced. ",
      "A merge is unnecessary; inspect the shard-processing results."
    )
  )
}

# Choose the first existing prefix as the merge base

base_idx    <- existing[1]
base_prefix <- subset_prefixes[base_idx]

# Everything else (excluding the base) goes into --pmerge-list

merge_targets <- subset_prefixes[setdiff(existing, base_idx)]
writeLines(merge_targets, con = merge_list)

merged_pfx <- file.path(tmp_dir, "ofh_subset_merged")

args_merge <- c(
  "--threads", threads,
  "--memory",  as.character(as.integer(mem_gb) * 1024),
  "--pfile",   base_prefix,        # <- now guaranteed to exist
  "--pmerge-list", merge_list,
  "--out",     merged_pfx
)

## remove any stale outputs

unlink(
  paste0(
    merged_pfx,
    c(".pgen", ".pvar", ".psam", ".log")
  ),
  force = TRUE
)

## run merge

message(sprintf(
  "Merging %d shard-level PGEN filesets into one genome-wide PGEN",
  length(existing)
))

merge_out <- system2(
  plink2_bin,
  args_merge,
  stdout = TRUE,
  stderr = TRUE
)


## check PLINK exit status and that the merged files exist

merge_status <- attr(
  merge_out,
  "status"
)

required_merged_files <- paste0(
  merged_pfx,
  c(".pgen", ".pvar", ".psam")
)

if (
  (!is.null(merge_status) && merge_status != 0L) ||
  !all(file.exists(required_merged_files))
) {
  cat(
    "---- PLINK merge output ----\n",
    paste(tail(merge_out, 100), collapse = "\n"),
    "\n",
    sep = ""
  )

  merge_log <- paste0(
    merged_pfx,
    ".log"
  )

  if (file.exists(merge_log)) {
    cat(
      "---- PLINK merge log tail ----\n",
      paste(tail(readLines(merge_log), 100), collapse = "\n"),
      "\n",
      sep = ""
    )
  }

  stop("PLINK shard merge failed.")
}



# This `--pmerge-list` route is the documented way to merge PLINK2 pfiles. [2](https://zenodo.org/records/15420125)

## Sanity check - how many variants carried over from HGDP+1KG loadings to OFH ?

# Overlap audit between loadings and merged .pfile (


# Harmonise HGDP+1KG reference IDs to OFH IDs 

library(data.table)

# Read the merged OFH PVAR
ofh_pvar <- fread(
  paste0(merged_pfx, ".pvar"),
  data.table = TRUE
)

required_pvar_columns <- c("#CHROM", "POS", "ID", "REF", "ALT")

missing_pvar_columns <- setdiff(
  required_pvar_columns,
  names(ofh_pvar)
)

if (length(missing_pvar_columns)) {
  stop(sprintf(
    "Merged PVAR is missing required column(s): %s",
    paste(missing_pvar_columns, collapse = ", ")
  ))
}

# Preserve the original OFH ID
ofh_pvar[, OFH_ID := as.character(ID)]

# Normalize chromosome and alleles
ofh_pvar[, CHROM_KEY := sub(
  "^chr",
  "",
  as.character(get("#CHROM")),
  ignore.case = TRUE
)]

ofh_pvar[, POS_KEY := as.character(POS)]
ofh_pvar[, REF_KEY := toupper(as.character(REF))]
ofh_pvar[, ALT_KEY := toupper(as.character(ALT))]

# Construct an order-independent chromosome-position-allele key
ofh_pvar[, ALLELE_LOW := ifelse(
  REF_KEY <= ALT_KEY,
  REF_KEY,
  ALT_KEY
)]

ofh_pvar[, ALLELE_HIGH := ifelse(
  REF_KEY <= ALT_KEY,
  ALT_KEY,
  REF_KEY
)]

ofh_pvar[, VARIANT_KEY := paste(
  CHROM_KEY,
  POS_KEY,
  ALLELE_LOW,
  ALLELE_HIGH,
  sep = ":"
)]

## reference loadings

reference_loadings <- fread(
  GRCh38_loadings_tsv,
  data.table = TRUE
)

required_loading_columns <- c(
  "ID",
  "ALT"
)

missing_loading_columns <- setdiff(
  required_loading_columns,
  names(reference_loadings)
)

if (length(missing_loading_columns)) {
  stop(sprintf(
    "GRCh38 loadings are missing required column(s): %s",
    paste(missing_loading_columns, collapse = ", ")
  ))
}

# Preserve the canonical reference ID
reference_loadings[, REFERENCE_ID := as.character(ID)]

loading_parts <- tstrsplit(
  reference_loadings$REFERENCE_ID,
  ":",
  fixed = TRUE
)

if (length(loading_parts) != 4L) {
  stop(
    "GRCh38 loading IDs are not consistently chr:position:REF:ALT."
  )
}

reference_loadings[, CHROM_KEY := sub(
  "^chr",
  "",
  loading_parts[[1]],
  ignore.case = TRUE
)]

reference_loadings[, POS_KEY := as.character(loading_parts[[2]])]
reference_loadings[, LOAD_REF := toupper(loading_parts[[3]])]
reference_loadings[, LOAD_ALT := toupper(loading_parts[[4]])]
reference_loadings[, ALT := toupper(as.character(ALT))]

reference_loadings[, ALLELE_LOW := ifelse(
  LOAD_REF <= LOAD_ALT,
  LOAD_REF,
  LOAD_ALT
)]

reference_loadings[, ALLELE_HIGH := ifelse(
  LOAD_REF <= LOAD_ALT,
  LOAD_ALT,
  LOAD_REF
)]

reference_loadings[, VARIANT_KEY := paste(
  CHROM_KEY,
  POS_KEY,
  ALLELE_LOW,
  ALLELE_HIGH,
  sep = ":"
)]

## detect ambiguous variants before joining

ambiguous_ofh_keys <- ofh_pvar[
  ,
  .(N_OFH_RECORDS = uniqueN(OFH_ID)),
  by = VARIANT_KEY
][
  N_OFH_RECORDS > 1L
]

if (nrow(ambiguous_ofh_keys)) {
  cat(
    "Examples of ambiguous OFH variant keys:\n"
  )

  print(
    head(
      ambiguous_ofh_keys,
      10L
    )
  )

  stop(sprintf(
    paste0(
      "%d chromosome-position-allele keys map to multiple OFH records. ",
      "These must be resolved before scoring."
    ),
    nrow(ambiguous_ofh_keys)
  ))
}

## build the ID map

id_map <- merge(
  reference_loadings[
    ,
    .(
      REFERENCE_ID,
      VARIANT_KEY,
      LOAD_REF,
      LOAD_ALT
    )
  ],
  ofh_pvar[
    ,
    .(
      OFH_ID,
      VARIANT_KEY,
      OFH_REF = REF_KEY,
      OFH_ALT = ALT_KEY
    )
  ],
  by = "VARIANT_KEY",
  all = FALSE
)

if (!nrow(id_map)) {
  stop(
    paste0(
      "No GRCh38 loading variants could be matched to the merged OFH PVAR ",
      "by chromosome, position, and alleles."
    )
  )
}

if (anyDuplicated(id_map$REFERENCE_ID)) {
  duplicated_reference_ids <- unique(
    id_map$REFERENCE_ID[
      duplicated(id_map$REFERENCE_ID)
    ]
  )

  stop(sprintf(
    "Some reference variants map to multiple OFH records, e.g. %s",
    paste(head(duplicated_reference_ids, 10L), collapse = ", ")
  ))
}

if (anyDuplicated(id_map$OFH_ID)) {
  duplicated_ofh_ids <- unique(
    id_map$OFH_ID[
      duplicated(id_map$OFH_ID)
    ]
  )

  stop(sprintf(
    "Some OFH variants map to multiple loading records, e.g. %s",
    paste(head(duplicated_ofh_ids, 10L), collapse = ", ")
  ))
}

## audit allele orientation

id_map[, ALLELE_RELATION := fcase(
  OFH_REF == LOAD_REF & OFH_ALT == LOAD_ALT,
  "same",

  OFH_REF == LOAD_ALT & OFH_ALT == LOAD_REF,
  "reversed",

  default = "mismatch"
)]

cat("==== OFH/reference allele audit ====\n")

print(
  id_map[
    ,
    .N,
    by = ALLELE_RELATION
  ]
)

if (any(id_map$ALLELE_RELATION == "mismatch")) {
  print(
    head(
      id_map[
        ALLELE_RELATION == "mismatch",
        .(
          REFERENCE_ID,
          OFH_ID,
          LOAD_REF,
          LOAD_ALT,
          OFH_REF,
          OFH_ALT
        )
      ],
      10L
    )
  )

  stop(
    "Some matched variants have incompatible allele sets."
  )
}

## validate the loadings "ALT" column

loading_alt_mismatch <- reference_loadings[
  is.na(ALT) |
  is.na(LOAD_ALT) |
  ALT != LOAD_ALT
]

if (nrow(loading_alt_mismatch)) {
  print(
    head(
      loading_alt_mismatch[, .(
        REFERENCE_ID,
        LOAD_REF,
        LOAD_ALT,
        ALT
      )],
      10L
    )
  )

  stop(sprintf(
    paste0(
      "%d reference-loading rows have an ALT column inconsistent ",
      "with the fourth component of ID."
    ),
    nrow(loading_alt_mismatch)
  ))
}

## create OFH-compatible loadings

mapped_loadings <- merge(
  reference_loadings,
  id_map[
    ,
    .(
      REFERENCE_ID,
      OFH_ID
    )
  ],
  by = "REFERENCE_ID",
  all = FALSE,
  sort = FALSE
)

# PLINK must see the OFH PGEN variant ID in the ID column
mapped_loadings[, ID := OFH_ID]

# Remove temporary parsing columns while retaining the original
# loading-file columns in their original order
original_loading_columns <- names(
  fread(
    GRCh38_loadings_tsv,
    nrows = 0L,
    data.table = TRUE
  )
)

mapped_loadings <- mapped_loadings[
  ,
  ..original_loading_columns
]

mapped_loadings_file <- file.path(
  tmp_dir,
  "HGDP_1KG_loadings_OFH_IDs.tsv"
)

fwrite(
  mapped_loadings,
  mapped_loadings_file,
  sep = "\t",
  quote = FALSE,
  na = "NA"
)

## create an OFH-compatible reference frequency file

reference_afreq <- fread(
  GRCh38_afreq_file,
  data.table = TRUE
)

# Normalize the variant-ID column name internally
if ("#ID" %in% names(reference_afreq)) {
  setnames(
    reference_afreq,
    old = "#ID",
    new = "ID"
  )
}

if (!"ID" %in% names(reference_afreq)) {
  stop(sprintf(
    "The GRCh38 allele-frequency file does not contain ID or #ID. Columns found: %s",
    paste(names(reference_afreq), collapse = ", ")
  ))
}

reference_afreq[, REFERENCE_ID := as.character(ID)]

required_afreq_columns <- c(
  "ID",
  "REF",
  "ALT",
  "ALT1_FREQ"
)

missing_afreq_columns <- setdiff(
  required_afreq_columns,
  names(reference_afreq)
)

if (length(missing_afreq_columns)) {
  stop(sprintf(
    "Reference frequency file is missing required column(s): %s",
    paste(missing_afreq_columns, collapse = ", ")
  ))
}

reference_afreq[, AFREQ_REF := toupper(
  as.character(REF)
)]

reference_afreq[, AFREQ_ALT := toupper(
  as.character(ALT)
)]

afreq_allele_check <- merge(
  reference_loadings[
    ,
    .(
      REFERENCE_ID,
      LOAD_REF,
      LOAD_ALT
    )
  ],
  reference_afreq[
    ,
    .(
      REFERENCE_ID,
      AFREQ_REF,
      AFREQ_ALT
    )
  ],
  by = "REFERENCE_ID",
  all = FALSE
)

afreq_allele_check[, ALLELE_RELATION := fcase(
  AFREQ_REF == LOAD_REF & AFREQ_ALT == LOAD_ALT,
  "same",

  AFREQ_REF == LOAD_ALT & AFREQ_ALT == LOAD_REF,
  "reversed",

  default = "mismatch"
)]

print(
  afreq_allele_check[
    ,
    .N,
    by = ALLELE_RELATION
  ]
)

if (any(afreq_allele_check$ALLELE_RELATION != "same")) {
  print(
    head(
      afreq_allele_check[
        ALLELE_RELATION != "same"
      ],
      10L
    )
  )

  stop(
    paste0(
      "Reference-frequency REF/ALT columns do not consistently match ",
      "the GRCh38 loading-file allele orientation."
    )
  )
}

mapped_afreq <- merge(
  reference_afreq,
  id_map[
    ,
    .(
      REFERENCE_ID,
      OFH_ID
    )
  ],
  by = "REFERENCE_ID",
  all = FALSE,
  sort = FALSE
)

mapped_afreq[, ID := OFH_ID]

original_afreq_columns <- names(
  fread(
    GRCh38_afreq_file,
    nrows = 0L,
    data.table = TRUE
  )
)

# Match the normalized internal column name
original_afreq_columns[
  original_afreq_columns == "#ID"
] <- "ID"

mapped_afreq <- mapped_afreq[
  ,
  ..original_afreq_columns
]

mapped_afreq_file <- file.path(
  tmp_dir,
  "HGDP_1KG_afreq_OFH_IDs.afreq"
)

## final mapping audit

n_reference_variants <- uniqueN(
  reference_loadings$REFERENCE_ID
)

n_ofh_variants <- uniqueN(
  ofh_pvar$OFH_ID
)

n_mapped_variants <- uniqueN(
  id_map$REFERENCE_ID
)

cat("==== Variant mapping summary ====\n")
cat(sprintf(
  "Unique HGDP+1KG loading variants: %d\n",
  n_reference_variants
))
cat(sprintf(
  "Unique variants in merged OFH PGEN: %d\n",
  n_ofh_variants
))
cat(sprintf(
  "HGDP+1KG variants mapped to OFH: %d\n",
  n_mapped_variants
))
cat(sprintf(
  "Loading variants absent from OFH: %d\n",
  n_reference_variants - n_mapped_variants
))
cat(sprintf(
  "Mapped proportion: %.2f%%\n",
  100 * n_mapped_variants / n_reference_variants
))

loading_ids <- unique(
  as.character(mapped_loadings$ID)
)

afreq_ids <- unique(
  as.character(mapped_afreq$ID)
)

missing_afreq_ids <- setdiff(
  loading_ids,
  afreq_ids
)

extra_afreq_ids <- setdiff(
  afreq_ids,
  loading_ids
)

if (length(missing_afreq_ids)) {
  print(
    head(
      missing_afreq_ids,
      10L
    )
  )

  stop(sprintf(
    paste0(
      "%d mapped loading variants do not have corresponding ",
      "HGDP+1KG reference-frequency records."
    ),
    length(missing_afreq_ids)
  ))
}

if (length(extra_afreq_ids)) {
  warning(sprintf(
    "%d mapped frequency records are absent from the mapped loadings.",
    length(extra_afreq_ids)
  ))
}

# Keep the same set of variants in both files
mapped_afreq <- mapped_afreq[
  ID %in% loading_ids
]

if (
  nrow(mapped_loadings) != nrow(mapped_afreq) ||
  !setequal(mapped_loadings$ID, mapped_afreq$ID)
) {
  stop(
    "Mapped loading and frequency files do not contain identical variant-ID sets."
  )
}

## confirm one row per ID

if (anyDuplicated(mapped_loadings$ID)) {
  stop("Duplicate variant IDs are present in mapped_loadings.")
}

if (anyDuplicated(mapped_afreq$ID)) {
  stop("Duplicate variant IDs are present in mapped_afreq.")
}

# Restore the original PLINK frequency-file header
data.table::setnames(
  mapped_afreq,
  old = "ID",
  new = "#ID"
)

fwrite(
  mapped_afreq,
  mapped_afreq_file,
  sep = "\t",
  quote = FALSE,
  na = "NA"
)

fwrite(
  id_map,
  file.path(
    tmp_dir,
    "HGDP_1KG_to_OFH_variant_mapping.tsv"
  ),
  sep = "\t",
  quote = FALSE
)



In [ ]:
# ========================================================== #
# Part 2C — Project OFH onto the HGDP+1KG reference PC axes  #
# ========================================================== #

library(data.table)

score_out <- file.path(
  tmp_dir,
  "OFH_on_HGDP1KG_GRCh38"
)

# Remove output from an earlier interrupted attempt
unlink(
  paste0(
    score_out,
    c(
      ".sscore",
      ".sscore.vars",
      ".log"
    )
  ),
  force = TRUE
)

args_score <- c(
  "--threads", threads,
  "--memory", as.character(as.integer(mem_gb) * 1024),

  "--pfile", merged_pfx,

  "--score",
  mapped_loadings_file,
  "variance-standardize",
  "cols=-scoreavgs,+scoresums",
  "header-read",
  "list-variants",

  "--score-col-nums", "3-22",

  "--read-freq",
  mapped_afreq_file,

  "--out",
  score_out
)

message(sprintf(
  paste0(
    "Projecting OFH onto PC1-PC20 using %d mapped ",
    "HGDP+1KG loading variants"
  ),
  nrow(mapped_loadings)
))

score_log <- system2(
  plink2_bin,
  args_score,
  stdout = TRUE,
  stderr = TRUE
)

score_status <- attr(
  score_log,
  "status"
)

required_score_files <- c(
  paste0(score_out, ".sscore"),
  paste0(score_out, ".sscore.vars")
)

if (
  (!is.null(score_status) && score_status != 0L) ||
  !all(file.exists(required_score_files))
) {
  cat(
    "---- PLINK scoring output ----\n",
    paste(tail(score_log, 100), collapse = "\n"),
    "\n",
    sep = ""
  )

  plink_score_log <- paste0(
    score_out,
    ".log"
  )

  if (file.exists(plink_score_log)) {
    cat(
      "---- PLINK scoring log tail ----\n",
      paste(
        tail(
          readLines(plink_score_log, warn = FALSE),
          100
        ),
        collapse = "\n"
      ),
      "\n",
      sep = ""
    )
  }

  stop(
    "PLINK HGDP+1KG PC projection failed."
  )
}

## identify variants used by PLINK

sscore_vars_file <- paste0(
  score_out,
  ".sscore.vars"
)

used_variant_table <- fread(
  sscore_vars_file,
  header = FALSE,
  data.table = TRUE
)

if (ncol(used_variant_table) != 1L) {
  stop(sprintf(
    "Expected one column in %s, but found %d.",
    sscore_vars_file,
    ncol(used_variant_table)
  ))
}

used_variant_ids <- unique(
  trimws(
    as.character(used_variant_table[[1L]])
  )
)

used_variant_ids <- used_variant_ids[
  !is.na(used_variant_ids) &
  nzchar(used_variant_ids) &
  !used_variant_ids %in% c("ID", "#ID")
]

used_variant_ids <- unique(
  as.character(used_variant_ids)
)

used_variant_ids <- used_variant_ids[
  !is.na(used_variant_ids) &
  nzchar(trimws(used_variant_ids)) &
  !used_variant_ids %in% c("ID", "#ID")
]

n_variants_used <- length(
  used_variant_ids
)

if (n_variants_used == 0L) {
  stop(
    "PLINK reported zero variants used for projection."
  )
}

pc_scale_factor <- sqrt(n_variants_used)

message(sprintf(
  "PLINK used %d unique variants for projection",
  n_variants_used
))

## record any variants that were requested but not used

mapped_variant_ids <- unique(
  as.character(
    mapped_loadings$ID
  )
)

mapped_but_not_used <- setdiff(
  mapped_variant_ids,
  used_variant_ids
)

message(sprintf(
  "%d mapped variants were not used by PLINK",
  length(mapped_but_not_used)
))

if (length(mapped_but_not_used)) {
  fwrite(
    data.table(
      ID = mapped_but_not_used
    ),
    file.path(
      tmp_dir,
      "HGDP_1KG_mapped_but_not_used.tsv"
    ),
    sep = "\t",
    quote = FALSE
  )
}

## calculate marker-retention statistics

mapping_rate <- (
  n_mapped_variants /
    n_reference_variants
)

scoring_retention_rate <- (
  n_variants_used /
    n_mapped_variants
)

overall_retention_rate <- (
  n_variants_used /
    n_reference_variants
)

cat("==== Projection marker retention ====\n")

cat(sprintf(
  "Reference loading variants: %d\n",
  n_reference_variants
))

cat(sprintf(
  "Reference variants mapped to OFH: %d\n",
  n_mapped_variants
))

cat(sprintf(
  "Variants used by PLINK: %d\n",
  n_variants_used
))

cat(sprintf(
  "Reference-to-OFH mapping rate: %.2f%%\n",
  100 * mapping_rate
))

cat(sprintf(
  "Mapped-to-scored retention rate: %.2f%%\n",
  100 * scoring_retention_rate
))

cat(sprintf(
  "Overall reference marker retention: %.2f%%\n",
  100 * overall_retention_rate
))

projection_audit <- data.table(
  reference_loading_variants = n_reference_variants,
  variants_mapped_to_OFH = n_mapped_variants,
  variants_used_by_PLINK = n_variants_used,
  pc_scale_factor = pc_scale_factor,
  mapping_rate = mapping_rate,
  scoring_retention_rate = scoring_retention_rate,
  overall_retention_rate = overall_retention_rate
)

projection_audit_file <- file.path(
  tmp_dir,
  "HGDP_1KG_OFH_projection_audit.tsv"
)

fwrite(
  projection_audit,
  projection_audit_file,
  sep = "\t",
  quote = FALSE
)

## parse raw PC score sums

ss <- fread(
  paste0(score_out, ".sscore"),
  data.table = TRUE
)

pc_sum <- intersect(
  paste0("PC", 1:20, "_SUM"),
  names(ss)
)

if (!length(pc_sum)) {
  pc_sum <- intersect(
    paste0("SCORE", 1:20, "_SUM"),
    names(ss)
  )
}

if (length(pc_sum) != 20L) {
  stop(sprintf(
    "Expected 20 PC score-sum columns, but found %d: %s",
    length(pc_sum),
    paste(pc_sum, collapse = ", ")
  ))
}

## validate the PC scaling denominator

if (!is.numeric(n_variants_used) ||
    length(n_variants_used) != 1L ||
    is.na(n_variants_used) ||
    n_variants_used <= 0L) {
  stop(
    "n_variants_used must be a single positive number before scaling."
  )
}

## create unscaled projected PCs from the raw PLINK score sums

ofh_projected_unscaled <- ss[
  ,
  c(
    list(IID = IID),
    setNames(
      lapply(
        pc_sum,
        function(column_name) {
          get(column_name)
        }
      ),
      sub(
        "^SCORE",
        "PC",
        sub(
          "_SUM$",
          "",
          pc_sum
        )
      )
    )
  )
]

## create a separate scaled copy

ofh_projected_scaled <- copy(
  ofh_projected_unscaled
)

pc_columns <- setdiff(
  names(ofh_projected_scaled),
  "IID"
)

ofh_projected_scaled[
  ,
  (pc_columns) := lapply(
    .SD,
    function(pc_value) {
      pc_value / pc_scale_factor
    }
  ),
  .SDcols = pc_columns
]

message(sprintf(
  paste0(
    "Retained unscaled PC score sums and created a scaled copy ",
    "using sqrt(%d) = %.6f, based on the variants reported ",
    "in .sscore.vars"
  ),
  n_variants_used,
  pc_scale_factor
))

## write unscaled and scaled projection outputs

ofh_projection_unscaled_file <- file.path(
  tmp_dir,
  "HGDP_1KG_PCs_OFH_projected_unscaled.csv"
)

ofh_projection_scaled_file <- file.path(
  tmp_dir,
  "HGDP_1KG_PCs_OFH_projected_scaled.csv"
)

fwrite(
  ofh_projected_unscaled,
  ofh_projection_unscaled_file
)

fwrite(
  ofh_projected_scaled,
  ofh_projection_scaled_file
)

message(sprintf(
  paste0(
    "Wrote unscaled and scaled OFH projections for %d participants ",
    "across %d PC axes"
  ),
  nrow(ofh_projected_unscaled),
  ncol(ofh_projected_unscaled) - 1L
))

## upload to persistent storage

mapping_file <- file.path(
  tmp_dir,
  "HGDP_1KG_to_OFH_variant_mapping.tsv"
)

mapped_not_used_file <- file.path(
  tmp_dir,
  "HGDP_1KG_mapped_but_not_used.tsv"
)

files_to_upload <- c(
  paste0(merged_pfx, ".pgen"),
  paste0(merged_pfx, ".pvar"),
  paste0(merged_pfx, ".psam"),
  paste0(merged_pfx, ".log"),

  mapped_loadings_file,
  mapped_afreq_file,
  mapping_file,
  projection_audit_file,

  paste0(score_out, ".sscore"),
  paste0(score_out, ".sscore.vars"),
  paste0(score_out, ".log"),

  ofh_projection_unscaled_file,
  ofh_projection_scaled_file
)

if (file.exists(mapped_not_used_file)) {
  files_to_upload <- c(
    files_to_upload,
    mapped_not_used_file
  )
}

files_to_upload <- unique(
  files_to_upload
)

for (file_to_upload in files_to_upload) {
  if (!file.exists(file_to_upload)) {
    warning(
      "Upload skipped because file is missing: ",
      file_to_upload
    )

    next
  }

  run_dx(c(
    "upload",
    file_to_upload
  ))
}